# I. OptiTrack

This section provides a concise guide on how to use **OptiTrack** for motion capture.
For more detailed information, refer to the official documentation:  
https://docs.optitrack.com/v3.0/motive

---

### 1. Camera Installation

- Ensure that all angles are well covered.
- Center the cameras on the robot.
- Avoid closing the diaphragm too much. Even if it introduces more noise, the markers will be significantly more visible.
- Ensure proper focus, especially for distant cameras.
- If a camera detects nothing:
  - First check the aperture.
  - If the issue persists, change the camera position.
- For calibration, strictly follow the instructions provided on the OptiTrack website.

---

### 2. Recording

- No specific constraints or special steps are required during recording.

---

### 3. Editing and Exporting

- Switch to **Edit Mode** by selecting the *Edit* layout.
- Select only the frames of interest (i.e., the frames during which the wing is moving).
- Create a **Rigid Body**:
  - Find a frame where all markers are visible.
  - Select the markers.
  - In the Builder panel, click **Create**.
- Label each marker using the format `Column–Row` (e.g., `A1`, `B3`, etc.), starting from `A1`.
  - This step must be done **manually**.
- Use **Auto-Label** to associate markers with each other (available in the *Label* window).
- Verify each marker trajectory for completeness and smoothness:
  - Click on a marker and inspect its trajectory in the **Graph View**.

#### Trajectory Corrections

- **Incomplete trajectories**:
  - Fill gaps using the *Cubic* method (recommended and generally more reliable).
  - Use the *Comparative* method only for large gaps, and select only the 2–3 closest markers.
- **Spikes or abnormal data**:
  - Cut the corrupted portion of the curve.
  - Reconstruct it using the *Cubic* method.
- **Smoothing**:
  - Apply smoothing to the trajectory.
  - Determine the appropriate smoothing frequency by testing multiple values.
  - The goal is a smooth trajectory without excessively deforming the wing shape or motion.

---

### 4. Exporting Data

- Export the labeled marker data to a CSV file:
  - Click **File → Export Tracking Data**.
  - Set the *Start Frame* and *Last Frame* to include only the frames of interest.
  - Export the data.


# II. Marker Placement

Correct marker placement is critical to ensure accurate reconstruction of the wing geometry and motion.

- Markers must be aligned **column-wise** along the span.
  - Perfect alignment in rows is **not required** and is not an issue.
- Place **at least two markers per column**.
- For each column:
  - Place **one marker on the leading edge**.
  - Place **one marker at the wingtip**.
  - Add the remaining markers afterward, distributed along the chord.
- Ensure all markers remain visible throughout the motion to avoid trajectory gaps.

### Example Setup

![Marker placement setup](setup.png)




# III. PteraSoftware

This section describes how to import **OptiTrack CSV data** into **PteraSoftware** and use it to perform aircraft simulations. An example implementation is available in:
`examples\Optitrack_unsteady_ring_vortex_lattice_method_solver_variable.py`

---

### 1. Create Variables

* Define the path to the OptiTrack CSV file:
```python
optitrack_file = r"C:\Users\henri\Documents\MIT\PteraSoftware\optitrack_data\four_hz.csv"
```

* Define the list of OptiTrack trackers:
```python
list_trackers = [
    "A1", "A2", "A3", "B1", "B2", "B3", "B4", "B5",
    "C1", "C2", "C3", "C4", "C5", "D1", "D2", "D3", "D4", "D5",
    "E1", "E2", "E3", "E4", "E5", "F1", "F2", "F3", "F4", "F5", 
    "G1", "G2", "G3", "G4", "G5", "H1", "H2", "H3", "H4",
    "I1", "I2"
]
```

* Define the flapping frequency:
```python 
frequency = 5.0  # Frequency of the flapping cycle in Hz.
```

* Extract CSV columns and load motion data:
```python
columns = ps.geometry.airfoil_creation.extract_columns(list_trackers)

data = ps.geometry.airfoil_creation.load_data(optitrack_file, list_trackers, right=False) * 10**-3  # Convert from mm to m
```
---

### 2. Create the Aircraft Geometry

```python
example_airplane = ps.geometry.airplane.Airplane(
    wings=[
        ps.geometry.wing.Wing(
            wing_cross_sections=ps.geometry.airfoil_creation.creation_wing_cross_sections(
                data, list_trackers, columns, frequency
            ),
            name="Main Wing",
            Ler_Gs_Cgs=np.array([0.0, 0.025, 0.0]),  # Position of the wing's rotation axis
            angles_Gs_to_Wn_ixyz=np.array([4, 0.0, 0.0]),  # Angle offset
            symmetric=True,
            mirror_only=False,
            symmetryNormal_G=(0.0, 0.0001, 0.0),
            symmetryPoint_G_Cg=(0.0, 0.0, 0.0),
            num_chordwise_panels=6,
            chordwise_spacing="uniform",
        )
    ],
    name="Example Airplane",
    Cg_GP1_CgP1=(0.0, 0.0, 0.0),  # Center of gravity (moments computed at this point)
    weight=5,
    s_ref=None,
    c_ref=None,
    b_ref=None,
)
```

* **Optional: Adding a Tail**

A tail can be added as follows:
```python
ps.geometry.wing.Wing(
            wing_cross_sections=[
                ps.geometry.wing_cross_section.WingCrossSection(
                    num_spanwise_panels=8,
                    chord=0.1,
                    Lp_Wcsp_Lpp=(0.0, 0.0, 0.0),
                    angles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
                    control_surface_symmetry_type="symmetric",
                    control_surface_hinge_point=0.75,
                    control_surface_deflection=0.0,
                    spanwise_spacing="uniform",
                    airfoil=ps.geometry.airfoil.Airfoil(
                        name="naca0012",
                        outline_A_lp=None,
                        resample=True,
                        n_points_per_side=400,
                    ),
                ),
                ps.geometry.wing_cross_section.WingCrossSection(
                    num_spanwise_panels=None,
                    chord=0.01,
                    Lp_Wcsp_Lpp=(0.09, 0.1, 0),
                    angles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
                    control_surface_symmetry_type="symmetric",
                    control_surface_hinge_point=0.75,
                    control_surface_deflection=0.0,
                    spanwise_spacing=None,
                    airfoil=ps.geometry.airfoil.Airfoil(
                        name="naca0012",
                        outline_A_lp=None,
                        resample=True,
                        n_points_per_side=400,
                    ),
                ),
            ],
            name="V-Tail",
            Ler_Gs_Cgs=(0.3, 0.0, 0.0),
            angles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
            symmetric=True,
            mirror_only=False,
            symmetryNormal_G=(0.0, 1.0, 0.0),
            symmetryPoint_G_Cg=(0.0, 0.0, 0.0),
            num_chordwise_panels=6,
            chordwise_spacing="uniform",
        ),
```
---

### 3. Define Wing Cross-Section Movements

* When the y-axis of the wing is not aligned with the normal axis of the symmetry plane, you must define the movement of both wings (the main one and the reflected one):
```python
main_wing_cross_section_movement=ps.geometry.airfoil_creation.wing_cross_sections_movement(example_airplane.wings[0], columns)

reflected_main_wing_cross_section_movement=ps.geometry.airfoil_creation.wing_cross_sections_movement(example_airplane.wings[1], columns)
```

* Tail Cross-Section Kinematics:
```python
v_tail_root_wing_cross_section_movement = (
     ps.movements.wing_cross_section_movement.WingCrossSectionMovement(
         base_wing_cross_section=example_airplane.wings[2].wing_cross_sections[0],
         ampLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         periodLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         spacingLp_Wcsp_Lpp=("sine", "sine", "sine"),
         phaseLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         ampAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
         periodAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
         spacingAngles_Wcsp_to_Wcs_ixyz=("sine", "sine", "sine"),
         phaseAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
     )
)
v_tail_tip_wing_cross_section_movement = (
     ps.movements.wing_cross_section_movement.WingCrossSectionMovement(
         base_wing_cross_section=example_airplane.wings[2].wing_cross_sections[1],
         ampLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         periodLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         spacingLp_Wcsp_Lpp=("sine", "sine", "sine"),
         phaseLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
         ampAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
         periodAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
         spacingAngles_Wcsp_to_Wcs_ixyz=("sine", "sine", "sine"),
         phaseAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
     )
)
```
---

### 4. Define Airplane Movement and Solver Setup

* Define the **WingMovements** for both wings by setting `optitrack=True`: 
```python
main_wing_movement = ps.movements.wing_movement.WingMovement(
    base_wing=example_airplane.wings[0],
    wing_cross_section_movements= main_wing_cross_section_movement,
    ampLer_Gs_Cgs=(0.0, 0.0, 0.0),
    periodLer_Gs_Cgs=(0.0, 0.0, 0.0),
    spacingLer_Gs_Cgs=("sine", "sine", "sine"),
    phaseLer_Gs_Cgs=(0.0, 0.0, 0.0),
    ampAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    periodAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    spacingAngles_Gs_to_Wn_ixyz=("sine", "sine", "sine"),
    phaseAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    optitrack = True  # Motion is fully driven by OptiTrack data
)
reflected_main_wing_movement = ps.movements.wing_movement.WingMovement(
    base_wing=example_airplane.wings[1],
    wing_cross_section_movements=reflected_main_wing_cross_section_movement,
    ampLer_Gs_Cgs=(0.0, 0.0, 0.0),
    periodLer_Gs_Cgs=(0.0, 0.0, 0.0),
    spacingLer_Gs_Cgs=("sine", "sine", "sine"),
    phaseLer_Gs_Cgs=(0.0, 0.0, 0.0),
    ampAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),  
    periodAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    spacingAngles_Gs_to_Wn_ixyz=("sine", "sine", "sine"),
    phaseAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    optitrack = True  # Motion is fully driven by OptiTrack data
)
```

* Tail Movement (If Present):
```python
v_tail_movement = ps.movements.wing_movement.WingMovement(
    base_wing=example_airplane.wings[2],
    wing_cross_section_movements=[v_tail_root_wing_cross_section_movement, v_tail_tip_wing_cross_section_movement],
    ampLer_Gs_Cgs=(0.0, 0.0, 0.0),
    periodLer_Gs_Cgs=(0.0, 0.0, 0.0),
    spacingLer_Gs_Cgs=("sine", "sine", "sine"),
    phaseLer_Gs_Cgs=(0.0, 0.0, 0.0),
    ampAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    periodAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    spacingAngles_Gs_to_Wn_ixyz=("sine", "sine", "sine"),
    phaseAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
)
```

* Define **AirplaneMovement**, **OperatingPoint**, and combine into a **Movement** object.
```python
example_airplane_movement = ps.movements.airplane_movement.AirplaneMovement(
    base_airplane=example_airplane,
    wing_movements=[main_wing_movement, reflected_main_wing_movement, v_tail_movement],
    ampCg_GP1_CgP1=(0.0, 0.0, 0.0),
    periodCg_GP1_CgP1=(0.0, 0.0, 0.0),
    spacingCg_GP1_CgP1=("sine", "sine", "sine"),
    phaseCg_GP1_CgP1=(0.0, 0.0, 0.0),
)

# Define the OperatingPoint. The operating point defines freestream conditions and flow properties:
# rho is the fluid's density
# vCg__E is the speed of the Airplane's or Airplanes' CG
# alpha is the angle of attack for the problem's Airplane(s)
# beta is the sideslip angle
# externalFX_W is the additional thrust or drag on a problem's Airplane(s)(in wind axes) not due to the Airplanes' Wings
# nu is the fluid's kinematic viscosity
example_operating_point = ps.operating_point.OperatingPoint(
    rho=1.225, vCg__E=5.0, alpha=12.0, beta=0.0, externalFX_W=0.0, nu=15.06e-6
)

# Define the operating point's OperatingPointMovement.
operating_point_movement = ps.movements.operating_point_movement.OperatingPointMovement(
    base_operating_point=example_operating_point,       
    ampVCg__E=0.0,
    periodVCg__E=0.0,
    spacingVCg__E="sine",
    phaseVCg__E=0.0,)

# Define the Movement. This contains the AirplaneMovement and the OperatingPointMovement.
movement = ps.movements.movement.Movement(
    airplane_movements=[example_airplane_movement],
    operating_point_movement=operating_point_movement,
    delta_time=1/360, # Sampling interval of OptiTrack frames
    num_cycles=None,
    num_chords=None,
    num_steps=50, # Ensure CSV duration is sufficient
)
```

* Define **UnsteadyProblem** and run the solver:
```python
example_problem = ps.problems.UnsteadyProblem(movement=movement)

example_solver = ps.unsteady_ring_vortex_lattice_method.UnsteadyRingVortexLatticeMethodSolver(
    unsteady_problem=example_problem
)

example_solver.run(
  prescribed_wake=True,
  calculate_streamlines=True,
  show_progress=True,
)
```

### 5. Plot Results

* Animate the airplane:

```python
ps.output.animate(
    unsteady_solver=example_solver,
    scalar_type="lift", # You can also choose "induced drag", "side force", or "lift"
    show_wake_vortices=True,
    save=True
)
```

![Animation](Animate.webp)

* Plot forces and moments over time:

```python
ps.output.plot_results_versus_time(unsteady_solver=example_solver, show=True)
```

![Forces Airplane](force_airplane.png)

* Plot wing loads over time:

```python
ps.output.plot_wing_loads_versus_time(unsteady_solver=example_solver, show=True)
```

![Moments Wing](moment_wing.png)

* Print total results:

```python
ps.output.print_results(example_solver)
```
```python
Airplane "Example Airplane":
  Final Forces (in wind axes):
    FX_W:      0.049 N     Drag:             -0.049 N
    FY_W:      -0.0 N      Side Force:       -0.0 N
    FZ_W:      -0.402 N    Lift:             0.402 N
  Final Moments (in wind axes, relative to the CG):
    MX_W_Cg:   -0.0 Nm     Rolling Moment:   -0.0 Nm
    MY_W_Cg:   -0.03 Nm    Pitching Moment:  -0.03 Nm
    MZ_W_Cg:   -0.0 Nm     Yawing Moment:    -0.0 Nm
  Final Force Coefficients (in wind axes):
    cFX_W:     0.06        CDi:              -0.06
    cFY_W:     -0.0        CY:               -0.0
    cFZ_W:     -0.498      CL:               0.498
  Final Moment Coefficients (in wind axes, relative to the CG):
    cMX_W_Cg:  -0.0        Cl:               -0.0
    cMY_W_Cg:  -0.251      Cm:               -0.251
    cMZ_W_Cg:  -0.0        Cn:               -0.0
```
---

### 6. Recreate the Same Airplane or Simulation

To validate simulation accuracy, the class `WingKinematicsComparison` in `optitrack_validation.py` enables reconstruction of the same geometry and motion using synthetic deformation models.

#### A. Geometry Replication

If you are only interested in a specific geometry (and do not want to manually create several wing cross sections), you can generate your airplane from the OptiTrack CSV file and then copy the geometry.
```python
simulated_airplane = ps.optitrack_validation.WingKinematicsComparison(
    example_airplane
).simulated_airplane
```

Then, create a simulation as usual. 
For FAAV-type configurations (rigid sections, deforming wing), cross-section motion can be defined as:
```python
wing_movements = []
for wing in simulated_airplane.wings:
    wing_cross_section_movements = []
    for wing_cross_section in wing.wing_cross_sections:
        wing_cross_section_movements.append(ps.movements.wing_cross_section_movement.WingCrossSectionMovement(
            base_wing_cross_section=wing_cross_section)
        )
    wing_movements.append(wing_cross_section_movements)
```

#### B. Recreating the Same Simulation

If you want to recreate the same simulation to compare simulated and real deformations, you can generate it directly using this class.
```python
WingKinematicsComparison = ps.optitrack_validation.WingKinematicsComparison(example_solver)
simulated_solver = WingKinematicsComparison.simulated_solver
```

Both solvers can then be executed for comparative analysis:
```python
example_solver.run(
    prescribed_wake=True,
    show_progress=True,
)
simulated_solver.run(
    prescribed_wake=True,
    show_progress=True,
)
```

**Warning** : 

* The current implementation supports a single aircraft with an optional tail.
  
* Certain parameters must be manually adjusted in the source code:
  * Mirror-wing symmetry logic (lines 140–143): 
  ```python 
  symmetric=True if len(self.base_airplane.wings) > 1 else False,    # We can't use real_wing.symmetric because after the creation of the reflected wing it becomes False (geometry.airplane, line 763)
  mirror_only=False,     # We can't use real_wing.mirror_only because after the creation of the reflected wing it becomes False (geometry.airplane, line 764)
  symmetryNormal_G=(0.0, 1.0, 0.0) if len(self.base_airplane.wings) > 1 else None,    # We can't use real_wing.symmetryNormal_G because after the creation of the reflected wing it becomes None (geometry.airplane, line 765)
  symmetryPoint_G_Cg=(0.0, 0.0, 0.0) if len(self.base_airplane.wings) > 1 else None,      # We can't use real_wing.symmetryPoint_G_Cg because after the creation of the reflected wing it becomes None (geometry.airplane, line 766)
  ```
  * Robot-specific body width offsets (lines 187–188): 
  ```python
  amplitudes_max.append(np.arctan2(p_max[2], p_max[1] - 25*10**-3))  # 25 is half the width of the robot body (in mm), change it for each robot
  amplitudes_min.append(np.arctan2(p_min[2], p_min[1] - 25*10**-3))
  ```

### 7. Comparison Tools for Simulated vs Real Wings
  
* Available analysis utilities include:
  
  * Dynamic wing comparison: `WingKinematicsComparison.dynamic_wing()`

![Dynamic](Dynamic.gif)

  * 3D trajectory of a point: `WingKinematicsComparison.plot_trajectory_3d(0.8, 0.5)`

![3d](3d_position.png)

  * Mean positional error vs time: `WingKinematicsComparison.plot_difference_position_versus_time()`

![Position_vs_time](position_vs_time.png)

  * Wing cross-section visualization: `WingKinematicsComparison.plot_section(5)`

![Section](Wing_section.png)

  * Panel-level force analysis: `WingKinematicsComparison.plot_panel_forces(0.8, 0.9)`

![Force\_panel](Force_panel.png)

  * Global force and moment comparison over time: `WingKinematicsComparison.plot_forces()`

![Force\_airplane\_dif](Force_airplane_dif.png)

  * Differential kinematic visualization:

```python
ps.output.animate(
    unsteady_solver=example_solver,
    scalar_type="difference position",
    show_wake_vortices=True,
    simulated_solver=simulated_solver,
    save=True,
)
```

![Differential](Difference.webp)
